# Check Mate — NLP-First Multimodal Handwriting Author Verification

**Purpose:** Align the Check Mate verification pipeline with the **IAM Handwriting Word Database** (`notebooks/Handwriting Word Database/`) and evaluate an **NLP-first, visually supported** authorship verification architecture.

### Architecture Overview
1. **IAM Dataset Loader & Page Reconstructor**: Indexes `words_new.txt` and `iam_words/words/` PNGs, reconstructs multi-line handwritten page halves (`partA` / `partB`) using true `(x, y, w, h)` bounding boxes, and builds `data/manifest.csv`.
2. **Handwriting OCR (Milestone 1)**: Segments pages into lines via OpenCV projection profiles and transcribes them with `microsoft/trocr-small-handwritten`, with CER/WER validation against ground-truth text.
3. **NLP Engine — Primary Component (Milestone 2)**: Extracts five stylometric modules via `spaCy` (`en_core_web_sm`), custom Python, and optional `flair/upos-english-fast` + `all-MiniLM-L6-v2`:
   - **Module A**: Lexical fingerprint (TTR, MATTR, word length, hapax ratio, function-word & stopword rates)
   - **Module B**: Syntactic fingerprint (POS distribution, noun-to-verb ratio, dependency depth, subordinate clauses)
   - **Module C**: Structural fingerprint (sentence length stats, line length, clause length, sentence openings)
   - **Module D**: Punctuation fingerprint (comma, semicolon, colon, parenthesis, question, exclamation rates)
   - **Module E**: Character-level linguistic patterns (double-letter rate, prefix/suffix rates, character n-gram TF-IDF cosine)
4. **Visual & Character-Level Profiling Engine — Supporting Component (Milestone 3)**:
   - OpenCV stroke width, slant angle, ink density, and connected-component proportions
   - Pretrained `ResNet18` line-crop embeddings + cosine similarity
   - Character-level prototype bank (`A–Z`, `a–z`, `0–9`) built only from observed characters (unobserved characters are explicitly marked `missing`)
5. **Writer-Disjoint Verification, Fusion & Ablation Study (Milestone 4)**:
   - Evaluates **NLP-only**, **Visual-only**, and **NLP + Visual Fusion** ($S_{\text{final}} = \alpha S_{\text{NLP}} + \beta S_{\text{Visual}}$, initial $\alpha=0.75, \beta=0.25$, plus validation-tuned fusion), reporting Accuracy, F1, ROC-AUC, **FAR, FRR, and EER**.
6. **Explainability, Insufficient-Evidence Guard & Artifact Export (Milestone 5)**:
   - Permutation importance, per-module similarity breakdown, minimum-word guard (`insufficient_evidence`), and export to `../backend/artifacts/`.

> **Important:** Check Mate estimates stylistic and handwriting compatibility. It does **not** prove identity or serve as forensic evidence. OCR errors, short text length, and topic shifts all affect scores.

## 1. Install dependencies

Run this cell once in your notebook environment and restart the kernel if prompted.
`sentencepiece`, `tiktoken`, and `protobuf` are required by `transformers` for `microsoft/trocr-small-handwritten`.

In [ ]:
%pip install -q transformers torch torchvision opencv-python pillow pandas numpy scikit-learn spacy matplotlib seaborn joblib tqdm sentencepiece tiktoken protobuf
!python -m spacy download en_core_web_sm

### Optional NLP Extensions (`flair` & `sentence-transformers`)
Uncomment and run if you want to enable `flair/upos-english-fast` and `sentence-transformers/all-MiniLM-L6-v2` experiments.

In [ ]:
# %pip install -q flair sentence-transformers

## 2. Imports and configuration

In [ ]:
from pathlib import Path
from collections import Counter
import json
import os
import re
import warnings

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    RocCurveDisplay
)
from sklearn.metrics.pairwise import paired_cosine_distances
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from joblib import dump

RANDOM_STATE = 42
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
TROCR_MODEL_NAME = "microsoft/trocr-small-handwritten"

# Resolve paths whether Jupyter runs inside `notebooks/` or at the repository root
CWD = Path.cwd().resolve()
NOTEBOOK_DIR = CWD if (CWD / "Handwriting Word Database").exists() else (CWD / "notebooks")
PROJECT_ROOT = NOTEBOOK_DIR.parent
IAM_DIR = NOTEBOOK_DIR / "Handwriting Word Database"
DATA_DIR = NOTEBOOK_DIR / "data"
DOCS_DIR = DATA_DIR / "iam_docs"
MANIFEST_PATH = DATA_DIR / "manifest.csv"
ARTIFACT_DIR = PROJECT_ROOT / "backend" / "artifacts"

# Experiment knobs
MAX_FORMS = None              # Set e.g. 120 for fast prototyping, or None to use all eligible IAM forms
MIN_LINES_PER_FORM = 6        # Split each form into >=3 lines per half (partA / partB)
MIN_WORDS_PER_FORM = 40
MIN_WORDS_FOR_VERIFICATION = 20
DEFAULT_ALPHA = 0.75          # Starting NLP weight
DEFAULT_BETA = 0.25           # Starting Visual weight
USE_FLAIR_UPOS = False        # Set True if `flair` is installed
USE_MINILM = False            # Set True if `sentence-transformers` is installed

print("Device:", DEVICE)
print("Notebook dir:", NOTEBOOK_DIR)
print("IAM database dir exists:", IAM_DIR.exists())
print("Artifact dir:", ARTIFACT_DIR)

## 3. IAM Handwriting Word Database Loader & Document Reconstructor

The local database in `notebooks/Handwriting Word Database/` contains:
- `words_new.txt` (and `iam_words/words.txt`): word IDs (`form-line-word`), segmentation status (`ok`/`err`), `(x, y, w, h)` coordinates, POS tag, and transcription.
- `iam_words/words/<prefix>/<form_id>/<word_id>.png`: cropped word images.

**Note on `writer_id`:** The official IAM `forms.txt` file maps `form_id -> writer_id`. If `forms.txt` is placed in `Handwriting Word Database/`, this loader uses true IAM `writer_id`s automatically. Otherwise, it safely groups by `form_id` and splits each form into non-overlapping top/bottom halves (`partA` and `partB`) so same-form pairs share a writer and train/val/test splits remain form-disjoint.

In [ ]:
def detokenize_iam_text(tokens: list[str]) -> str:
    """Normalize IAM word-token spacing around punctuation and contractions."""
    text = " ".join(str(t) for t in tokens)
    text = re.sub(r"\s+([.,;:?!%)\]}])", r"\1", text)
    text = re.sub(r"([(\[{])\s+", r"\1", text)
    text = re.sub(r"\s+('(s|t|m|re|ve|ll|d)\b)", r"\1", text, flags=re.IGNORECASE)
    text = re.sub(r"\bn\s+'t\b", "n't", text, flags=re.IGNORECASE)
    return text.strip()


def load_iam_word_index(iam_dir: Path = IAM_DIR) -> pd.DataFrame:
    """Parse IAM words metadata and match against existing word PNG files."""
    words_file = iam_dir / "words_new.txt"
    if not words_file.exists():
        words_file = iam_dir / "iam_words" / "words.txt"
    if not words_file.exists():
        raise FileNotFoundError(f"Could not find words_new.txt or words.txt in {iam_dir}")

    img_root = iam_dir / "iam_words" / "words"
    form_to_writer = {}
    for cand in [iam_dir / "forms.txt", iam_dir / "ascii" / "forms.txt"]:
        if cand.exists():
            for raw in cand.read_text(encoding="utf-8", errors="ignore").splitlines():
                line = raw.strip()
                if not line or line.startswith("#"):
                    continue
                cols = line.split()
                if len(cols) >= 2:
                    form_to_writer[cols[0]] = cols[1]
            break

    rows = []
    for raw in words_file.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split(maxsplit=8)
        if len(parts) < 9:
            continue
        wid, status, gray, x, y, w, h, tag, text = parts
        try:
            form_id, line_no, word_no = wid.rsplit("-", 2)
        except ValueError:
            continue
        prefix = form_id.split("-")[0]
        img_path = img_root / prefix / form_id / f"{wid}.png"
        if not img_path.exists():
            continue
        rows.append({
            "word_id": wid,
            "form_id": form_id,
            "writer_id": form_to_writer.get(form_id, form_id),
            "writer_id_source": "forms.txt" if form_id in form_to_writer else "form_id_fallback",
            "passage_id": form_id.rstrip("ux"),
            "line_no": int(line_no),
            "word_no": int(word_no),
            "status": status,
            "x": int(x),
            "y": int(y),
            "w": int(w),
            "h": int(h),
            "tag": tag,
            "text": text,
            "image_path": str(img_path),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("No valid IAM word images matched words_new.txt.")
    return df


def reconstruct_document_image(word_df: pd.DataFrame, pad: int = 30) -> np.ndarray:
    """Render word crops onto a white canvas at their original (x, y, w, h) coordinates."""
    min_x = int(word_df["x"].min())
    min_y = int(word_df["y"].min())
    max_x = int((word_df["x"] + word_df["w"]).max())
    max_y = int((word_df["y"] + word_df["h"]).max())

    canvas_h = max(64, max_y - min_y + 2 * pad)
    canvas_w = max(64, max_x - min_x + 2 * pad)
    canvas = np.full((canvas_h, canvas_w, 3), 255, dtype=np.uint8)

    for _, row in word_df.iterrows():
        img = cv2.imread(row["image_path"])
        if img is None:
            continue
        ih, iw = img.shape[:2]
        ry = max(0, int(row["y"]) - min_y + pad)
        rx = max(0, int(row["x"]) - min_x + pad)
        ry2 = min(canvas.shape[0], ry + ih)
        rx2 = min(canvas.shape[1], rx + iw)
        if ry2 > ry and rx2 > rx:
            canvas[ry:ry2, rx:rx2] = np.minimum(canvas[ry:ry2, rx:rx2], img[:ry2 - ry, :rx2 - rx])
    return canvas


def build_iam_document_manifest(
    df_words: pd.DataFrame,
    docs_dir: Path = DOCS_DIR,
    manifest_path: Path = MANIFEST_PATH,
    max_forms: int | None = MAX_FORMS,
    min_lines: int = MIN_LINES_PER_FORM,
    min_words: int = MIN_WORDS_PER_FORM,
    overwrite_images: bool = False,
) -> pd.DataFrame:
    """Split each IAM form into partA/partB multi-line document images and save manifest.csv."""
    docs_dir.mkdir(parents=True, exist_ok=True)
    manifest_path.parent.mkdir(parents=True, exist_ok=True)

    doc_rows = []
    form_ids = sorted(df_words["form_id"].unique())
    processed_forms = 0

    for form_id in form_ids:
        fdf = df_words[df_words["form_id"] == form_id]
        lines = sorted(fdf["line_no"].unique())
        if len(lines) < min_lines or len(fdf) < min_words:
            continue

        mid = len(lines) // 2
        halves = [("A", lines[:mid]), ("B", lines[mid:])]
        for part, line_subset in halves:
            sub = fdf[fdf["line_no"].isin(line_subset)].sort_values(["line_no", "word_no"])
            line_texts = [
                detokenize_iam_text(ldf["text"].tolist())
                for _, ldf in sub.groupby("line_no", sort=True)
            ]
            doc_text = "\n".join(t for t in line_texts if t)
            out_path = docs_dir / f"{form_id}_part{part}.png"
            if overwrite_images or not out_path.exists():
                canvas = reconstruct_document_image(sub)
                cv2.imwrite(str(out_path), canvas)

            doc_rows.append({
                "doc_id": f"{form_id}_part{part}",
                "image_path": str(out_path),
                "form_id": form_id,
                "writer_id": sub["writer_id"].iloc[0],
                "writer_id_source": sub["writer_id_source"].iloc[0],
                "passage_id": sub["passage_id"].iloc[0],
                "part": part,
                "line_count": len(line_subset),
                "word_count_gt": len(sub),
                "text": doc_text,
            })

        processed_forms += 1
        if max_forms is not None and processed_forms >= max_forms:
            break

    manifest_df = pd.DataFrame(doc_rows)
    manifest_df.to_csv(manifest_path, index=False)
    return manifest_df


df_words = load_iam_word_index(IAM_DIR)
print(f"Indexed {len(df_words):,} IAM word crops across {df_words['form_id'].nunique()} forms.")
print("Writer ID source:", df_words["writer_id_source"].iloc[0])

manifest = build_iam_document_manifest(df_words)
print(f"Built manifest with {len(manifest)} document halves across {manifest['writer_id'].nunique()} writers/forms.")
manifest.head(4)

In [ ]:
# Quick visual inspection of a reconstructed IAM document pair (same form: partA vs partB)
sample_a = manifest.iloc[0]
sample_b = manifest.iloc[1]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, row in zip(axes, [sample_a, sample_b]):
    img = cv2.imread(row["image_path"])
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{row['doc_id']} | writer={row['writer_id']} | lines={row['line_count']}")
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Reference GT text:\n", sample_a["text"])

## 4. Image Preprocessing, Line Segmentation & Handwriting OCR (TrOCR)

TrOCR (`microsoft/trocr-small-handwritten`) operates on **individual text lines**.
We segment each uploaded or reconstructed page into line crops using an adaptive-threshold horizontal projection profile, transcribe each line, and validate OCR quality against IAM ground truth using CER and WER.

In [ ]:
_trocr_processor = None
_trocr_model = None

def load_trocr(model_name: str = TROCR_MODEL_NAME):
    """Lazy-load TrOCR processor and model with clear dependency diagnostics."""
    global _trocr_processor, _trocr_model
    if _trocr_processor is not None and _trocr_model is not None:
        return _trocr_processor, _trocr_model
    try:
        _trocr_processor = TrOCRProcessor.from_pretrained(model_name)
        _trocr_model = VisionEncoderDecoderModel.from_pretrained(model_name).to(DEVICE)
        _trocr_model.eval()
        return _trocr_processor, _trocr_model
    except Exception as exc:
        raise RuntimeError(
            f"Failed to load {model_name}. Ensure `sentencepiece`, `tiktoken`, and `protobuf` are installed "
            f"(run Cell 1 and restart the kernel if needed). Original error: {exc}"
        ) from exc


def load_page(image_path: str | Path) -> np.ndarray:
    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
    return image


def preprocess_page(image: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image.copy()
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    binary = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 31, 15
    )
    return binary


def segment_lines(image: np.ndarray, min_line_height: int = 12, padding: int = 6) -> list[np.ndarray]:
    """Return cropped text-line images in top-to-bottom order."""
    binary = preprocess_page(image)
    row_ink = np.sum(binary > 0, axis=1)
    active = row_ink > max(2, int(binary.shape[1] * 0.003))

    ranges = []
    start = None
    for idx, is_active in enumerate(active):
        if is_active and start is None:
            start = idx
        elif not is_active and start is not None:
            if idx - start >= min_line_height:
                ranges.append((start, idx))
            start = None
    if start is not None and len(active) - start >= min_line_height:
        ranges.append((start, len(active)))

    crops = []
    for y1, y2 in ranges:
        y1 = max(0, y1 - padding)
        y2 = min(image.shape[0], y2 + padding)
        crop = image[y1:y2, :]
        if crop.size:
            crops.append(crop)
    return crops


def transcribe_line(line_image: np.ndarray) -> str:
    processor, trocr = load_trocr()
    if line_image.ndim == 2:
        pil_image = Image.fromarray(line_image).convert("RGB")
    else:
        pil_image = Image.fromarray(cv2.cvtColor(line_image, cv2.COLOR_BGR2RGB))
    pixel_values = processor(images=pil_image, return_tensors="pt").pixel_values.to(DEVICE)
    with torch.no_grad():
        generated_ids = trocr.generate(pixel_values, max_new_tokens=128)
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()


def transcribe_page(image_path: str | Path, show_lines: bool = False) -> dict:
    page = load_page(image_path)
    line_images = segment_lines(page)
    if not line_images:
        return {"text": "", "line_texts": [], "line_count": 0, "line_images": []}

    if show_lines:
        fig, axes = plt.subplots(len(line_images), 1, figsize=(12, max(2, 1.5 * len(line_images))))
        if len(line_images) == 1:
            axes = [axes]
        for ax, line in zip(axes, line_images):
            ax.imshow(cv2.cvtColor(line, cv2.COLOR_BGR2RGB))
            ax.axis("off")
        plt.tight_layout()
        plt.show()

    line_texts = [transcribe_line(line) for line in line_images]
    return {
        "text": "\n".join(t for t in line_texts if t),
        "line_texts": line_texts,
        "line_count": len(line_images),
        "line_images": line_images,
    }


def _levenshtein(seq_a, seq_b) -> int:
    m, n = len(seq_a), len(seq_b)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            cost = 0 if seq_a[i - 1] == seq_b[j - 1] else 1
            prev, dp[j] = dp[j], min(dp[j] + 1, dp[j - 1] + 1, prev + cost)
    return dp[n]


def evaluate_ocr_sample(manifest_df: pd.DataFrame, n_samples: int = 4) -> pd.DataFrame:
    """Compare TrOCR transcriptions against IAM ground-truth text on a small sample."""
    rows = []
    for _, row in manifest_df.head(n_samples).iterrows():
        ocr_res = transcribe_page(row["image_path"])
        gt = row["text"].replace("\n", " ").strip()
        pred = ocr_res["text"].replace("\n", " ").strip()
        cer = _levenshtein(gt, pred) / max(1, len(gt))
        wer = _levenshtein(gt.split(), pred.split()) / max(1, len(gt.split()))
        rows.append({
            "doc_id": row["doc_id"],
            "gt_lines": row["line_count"],
            "detected_lines": ocr_res["line_count"],
            "cer": round(cer, 4),
            "wer": round(wer, 4),
            "gt_preview": gt[:80],
            "ocr_preview": pred[:80],
        })
    return pd.DataFrame(rows)

# Uncomment to benchmark TrOCR after installing `sentencepiece` / `tiktoken`:
# ocr_eval_df = evaluate_ocr_sample(manifest, n_samples=4)
# display(ocr_eval_df)

## 5. NLP Engine — Five Stylometric Modules (Primary Component)

We implement all five linguistic modules from the Check Mate specification:
- **Module A (Lexical)**: TTR, MATTR-50, word length mean/std, hapax ratio, lexical density, stopword rate, function-word rate, plus 67 individual function-word frequencies.
- **Module B (Syntactic)**: spaCy POS distribution, noun-to-verb ratio, dependency depth, subordinate clause rate (plus optional `flair/upos-english-fast`).
- **Module C (Structural)**: sentence length mean/std/variance, average line length, average clause length, complex sentence ratio, and sentence-opening POS rates.
- **Module D (Punctuation)**: punctuation density plus comma, semicolon, colon, question, exclamation, parenthesis, period, and quote rates.
- **Module E (Character-level linguistic patterns)**: double-letter rate, common prefix/suffix rates, and TF-IDF character n-grams (`3–5`).

Note: `word_count`, `sentence_count`, and `line_count` are kept as **metadata** for the `insufficient_evidence` guard rather than fed to the classifier, preventing trivial length leakage.

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["ner"])

FUNCTION_WORDS = [
    "a", "an", "the", "and", "or", "but", "if", "because", "as", "of",
    "at", "by", "for", "from", "in", "into", "on", "onto", "to", "with",
    "without", "is", "am", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "i", "you", "he", "she",
    "it", "we", "they", "me", "him", "her", "us", "them", "my", "your",
    "our", "their", "this", "that", "these", "those", "not", "no", "so",
    "than", "then", "there", "here", "which", "who", "what", "when",
    "where", "why", "how"
]

NLP_MODULE_MAP = {
    "lexical": [
        "avg_word_length", "std_word_length", "ttr", "mattr_50",
        "hapax_ratio", "lexical_density", "stopword_rate", "function_word_rate",
        *[f"fw_{w}" for w in FUNCTION_WORDS],
    ],
    "syntactic": [
        "noun_rate", "verb_rate", "adj_rate", "adv_rate", "pron_rate",
        "adp_rate", "det_rate", "conj_rate", "noun_to_verb_ratio",
        "dependency_depth_mean", "subordinate_clause_rate",
    ],
    "structural": [
        "avg_sentence_length", "std_sentence_length", "var_sentence_length",
        "avg_line_length", "avg_clause_length", "complex_sentence_ratio",
        "open_pron_rate", "open_det_rate", "open_conj_rate", "open_adv_rate",
    ],
    "punctuation": [
        "punct_per_token", "comma_rate", "semicolon_rate", "colon_rate",
        "question_rate", "exclamation_rate", "parenthesis_rate",
        "period_rate", "quote_rate",
    ],
    "character_nlp": [
        "double_letter_rate", "suffix_ing_rate", "suffix_ed_rate",
        "suffix_ly_rate", "suffix_tion_rate", "prefix_un_rate",
        "prefix_re_rate", "prefix_in_rate",
    ],
}

ALL_NLP_FEATURE_COLS = [col for cols in NLP_MODULE_MAP.values() for col in cols]
NLP_META_COLS = ["meta_word_count", "meta_sentence_count", "meta_line_count"]


def moving_average_ttr(words: list[str], window: int = 50) -> float:
    if len(words) < 2:
        return 0.0
    if len(words) <= window:
        return len(set(words)) / len(words)
    scores = [
        len(set(words[i:i + window])) / window
        for i in range(0, len(words) - window + 1, max(1, window // 2))
    ]
    return float(np.mean(scores)) if scores else 0.0


def extract_stylometric_features(text: str, line_count: int | None = None) -> dict:
    clean_text = text or ""
    lines = [ln for ln in clean_text.splitlines() if ln.strip()]
    n_lines = line_count if line_count is not None else max(1, len(lines))

    doc = nlp(clean_text)
    words = [t.text.lower() for t in doc if t.is_alpha]
    sentences = list(doc.sents)
    n_words = len(words)
    n_tokens = len(doc)
    n_sentences = len(sentences)

    counts = Counter(words)
    pos_counts = Counter(t.pos_ for t in doc if not t.is_space)
    punct_counts = Counter(t.text for t in doc if t.is_punct)
    function_counts = Counter(w for w in words if w in FUNCTION_WORDS)

    sentence_lengths = [sum(1 for t in sent if t.is_alpha) for sent in sentences]
    word_lengths = [len(w) for w in words]
    total_punct = sum(punct_counts.values())
    sub_dep_set = {"advcl", "ccomp", "csubj", "acl", "relcl"}
    sub_clauses = sum(1 for t in doc if t.dep_ in sub_dep_set)

    # Complex sentences: sentences containing at least one subordinate clause
    complex_sents = sum(
        1 for sent in sentences if any(t.dep_ in sub_dep_set for t in sent)
    )

    # Sentence openings (first non-space, non-punct token POS)
    open_pos = []
    for sent in sentences:
        first = next((t for t in sent if not t.is_space and not t.is_punct), None)
        if first is not None:
            open_pos.append(first.pos_)
    open_counts = Counter(open_pos)
    n_open = max(1, len(open_pos))

    noun_count = pos_counts["NOUN"] + pos_counts["PROPN"]
    verb_count = pos_counts["VERB"] + pos_counts["AUX"]

    features = {
        # Metadata (for insufficient-evidence guard, excluded from classifier features)
        "meta_word_count": n_words,
        "meta_sentence_count": n_sentences,
        "meta_line_count": n_lines,

        # Module A: Lexical
        "avg_word_length": float(np.mean(word_lengths)) if word_lengths else 0.0,
        "std_word_length": float(np.std(word_lengths)) if word_lengths else 0.0,
        "ttr": len(set(words)) / n_words if n_words else 0.0,
        "mattr_50": moving_average_ttr(words, 50),
        "hapax_ratio": sum(1 for c in counts.values() if c == 1) / n_words if n_words else 0.0,
        "lexical_density": sum(1 for t in doc if t.pos_ in {"NOUN", "VERB", "ADJ", "ADV"}) / n_tokens if n_tokens else 0.0,
        "stopword_rate": sum(1 for t in doc if t.is_alpha and t.is_stop) / n_words if n_words else 0.0,
        "function_word_rate": sum(function_counts.values()) / n_words if n_words else 0.0,

        # Module B: Syntactic
        "noun_rate": pos_counts["NOUN"] / n_tokens if n_tokens else 0.0,
        "verb_rate": pos_counts["VERB"] / n_tokens if n_tokens else 0.0,
        "adj_rate": pos_counts["ADJ"] / n_tokens if n_tokens else 0.0,
        "adv_rate": pos_counts["ADV"] / n_tokens if n_tokens else 0.0,
        "pron_rate": pos_counts["PRON"] / n_tokens if n_tokens else 0.0,
        "adp_rate": pos_counts["ADP"] / n_tokens if n_tokens else 0.0,
        "det_rate": pos_counts["DET"] / n_tokens if n_tokens else 0.0,
        "conj_rate": (pos_counts["CCONJ"] + pos_counts["SCONJ"]) / n_tokens if n_tokens else 0.0,
        "noun_to_verb_ratio": noun_count / max(1, verb_count),
        "dependency_depth_mean": float(np.mean([
            max((len(list(t.ancestors)) for t in sent), default=0)
            for sent in sentences
        ])) if sentences else 0.0,
        "subordinate_clause_rate": sub_clauses / n_sentences if n_sentences else 0.0,

        # Module C: Structural
        "avg_sentence_length": float(np.mean(sentence_lengths)) if sentence_lengths else 0.0,
        "std_sentence_length": float(np.std(sentence_lengths)) if sentence_lengths else 0.0,
        "var_sentence_length": float(np.var(sentence_lengths)) if sentence_lengths else 0.0,
        "avg_line_length": n_words / max(1, n_lines),
        "avg_clause_length": n_words / max(1, n_sentences + sub_clauses),
        "complex_sentence_ratio": complex_sents / n_sentences if n_sentences else 0.0,
        "open_pron_rate": open_counts["PRON"] / n_open,
        "open_det_rate": open_counts["DET"] / n_open,
        "open_conj_rate": (open_counts["CCONJ"] + open_counts["SCONJ"]) / n_open,
        "open_adv_rate": open_counts["ADV"] / n_open,

        # Module D: Punctuation
        "punct_per_token": total_punct / n_tokens if n_tokens else 0.0,
        "comma_rate": punct_counts[","] / total_punct if total_punct else 0.0,
        "semicolon_rate": punct_counts[";"] / total_punct if total_punct else 0.0,
        "colon_rate": punct_counts[":"] / total_punct if total_punct else 0.0,
        "question_rate": punct_counts["?"] / total_punct if total_punct else 0.0,
        "exclamation_rate": punct_counts["!"] / total_punct if total_punct else 0.0,
        "parenthesis_rate": (punct_counts["("] + punct_counts[")"]) / total_punct if total_punct else 0.0,
        "period_rate": punct_counts["."] / total_punct if total_punct else 0.0,
        "quote_rate": (punct_counts['"'] + punct_counts["'"]) / total_punct if total_punct else 0.0,

        # Module E: Character-level linguistic patterns
        "double_letter_rate": sum(1 for w in words if re.search(r"([a-z])\1", w)) / n_words if n_words else 0.0,
        "suffix_ing_rate": sum(1 for w in words if w.endswith("ing")) / n_words if n_words else 0.0,
        "suffix_ed_rate": sum(1 for w in words if w.endswith("ed")) / n_words if n_words else 0.0,
        "suffix_ly_rate": sum(1 for w in words if w.endswith("ly")) / n_words if n_words else 0.0,
        "suffix_tion_rate": sum(1 for w in words if w.endswith(("tion", "sion"))) / n_words if n_words else 0.0,
        "prefix_un_rate": sum(1 for w in words if len(w) > 3 and w.startswith("un")) / n_words if n_words else 0.0,
        "prefix_re_rate": sum(1 for w in words if len(w) > 3 and w.startswith("re")) / n_words if n_words else 0.0,
        "prefix_in_rate": sum(1 for w in words if len(w) > 3 and w.startswith(("in", "im"))) / n_words if n_words else 0.0,
    }

    for word in FUNCTION_WORDS:
        features[f"fw_{word}"] = counts[word] / n_words if n_words else 0.0

    return features


def build_char_vectorizer() -> TfidfVectorizer:
    return TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        min_df=2,
        max_features=5000,
        sublinear_tf=True,
        norm="l2",
    )


def compute_nlp_module_similarities(
    ref_features: dict,
    cmp_features: dict,
    char_ngram_cosine: float | None = None,
) -> dict[str, float]:
    """Compute normalized [0, 1] similarity breakdown across the 5 NLP modules."""
    scores = {}
    for module_name, cols in NLP_MODULE_MAP.items():
        diffs = []
        for c in cols:
            a, b = float(ref_features.get(c, 0.0)), float(cmp_features.get(c, 0.0))
            denom = abs(a) + abs(b) + 1e-6
            diffs.append(abs(a - b) / denom)
        sim = float(np.clip(1.0 - np.mean(diffs), 0.0, 1.0))
        if module_name == "character_nlp" and char_ngram_cosine is not None:
            sim = float(0.5 * sim + 0.5 * np.clip(char_ngram_cosine, 0.0, 1.0))
        scores[module_name] = round(sim, 4)
    return scores


# Optional Flair UPOS & MiniLM hooks
_flair_tagger = None
_minilm_model = None

def extract_flair_upos_rates(text: str) -> dict[str, float]:
    global _flair_tagger
    if not USE_FLAIR_UPOS:
        return {}
    try:
        from flair.models import SequenceTagger
        from flair.data import Sentence
        if _flair_tagger is None:
            _flair_tagger = SequenceTagger.load("flair/upos-english-fast")
        sent = Sentence(text or "")
        _flair_tagger.predict(sent)
        tags = Counter(tok.get_label("upos").value for tok in sent)
        total = max(1, len(sent))
        return {f"flair_upos_{k}": v / total for k, v in tags.items()}
    except Exception as exc:
        warnings.warn(f"Flair UPOS skipped: {exc}")
        return {}


def compute_semantic_similarity(text_a: str, text_b: str) -> float | None:
    global _minilm_model
    if not USE_MINILM:
        return None
    try:
        from sentence_transformers import SentenceTransformer
        if _minilm_model is None:
            _minilm_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        emb = _minilm_model.encode([text_a, text_b], normalize_embeddings=True)
        return float(np.dot(emb[0], emb[1]))
    except Exception as exc:
        warnings.warn(f"MiniLM semantic similarity skipped: {exc}")
        return None

## 6. Visual Handwriting & Character-Level Profiling Engine (Supporting Component)

This branch supports the NLP engine using:
1. **OpenCV handwriting & stroke descriptors**: stroke width (distance transform), slant angle & consistency (Sobel stroke gradients), ink density, and connected-component proportions.
2. **Pretrained `ResNet18` visual encoder**: extracts 512-D L2-normalized visual embeddings averaged across segmented text lines.
3. **Character-Level Prototype Bank (`A–Z`, `a–z`, `0–9`)**: builds per-character prototypes from observed character/short-word crops in IAM and explicitly marks unseen characters as `missing`.

In [ ]:
def extract_visual_handwriting_features(image: np.ndarray) -> dict[str, float]:
    """Extract stroke, slant, ink density, and connected-component geometry from a page image."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image.copy()
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    ink_mask = binary > 0
    ink_density = float(np.mean(ink_mask))

    dist = cv2.distanceTransform(binary, cv2.DIST_L2, 3)
    fg_dist = dist[ink_mask]
    stroke_width_mean = float(np.mean(fg_dist) * 2.0) if fg_dist.size else 0.0
    stroke_width_std = float(np.std(fg_dist) * 2.0) if fg_dist.size else 0.0

    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.hypot(gx, gy)
    strong = (mag > np.percentile(mag, 85)) & ink_mask if np.any(ink_mask) else np.zeros_like(ink_mask)
    if np.any(strong):
        angles = np.degrees(np.arctan2(gy[strong], gx[strong]))
        slant_vals = ((angles + 90.0) % 180.0) - 90.0
        slant_angle_mean = float(np.mean(slant_vals))
        slant_angle_std = float(np.std(slant_vals))
    else:
        slant_angle_mean, slant_angle_std = 0.0, 0.0

    _, _, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    valid = stats[1:]
    valid = valid[valid[:, cv2.CC_STAT_AREA] >= 12] if len(valid) else valid
    if len(valid):
        heights = valid[:, cv2.CC_STAT_HEIGHT]
        widths = valid[:, cv2.CC_STAT_WIDTH]
        aspects = widths / np.maximum(heights, 1)
        cc_height_mean = float(np.mean(heights))
        cc_height_std = float(np.std(heights))
        cc_aspect_mean = float(np.mean(aspects))
    else:
        cc_height_mean = cc_height_std = cc_aspect_mean = 0.0

    return {
        "vis_ink_density": ink_density,
        "vis_stroke_width_mean": stroke_width_mean,
        "vis_stroke_width_std": stroke_width_std,
        "vis_slant_angle_mean": slant_angle_mean,
        "vis_slant_angle_std": slant_angle_std,
        "vis_cc_height_mean": cc_height_mean,
        "vis_cc_height_std": cc_height_std,
        "vis_cc_aspect_mean": cc_aspect_mean,
    }


VISUAL_FEATURE_COLS = [
    "vis_ink_density", "vis_stroke_width_mean", "vis_stroke_width_std",
    "vis_slant_angle_mean", "vis_slant_angle_std",
    "vis_cc_height_mean", "vis_cc_height_std", "vis_cc_aspect_mean",
]

_resnet_encoder = None
_resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def load_resnet18_encoder() -> nn.Module:
    """Load ResNet18 truncated before the final FC layer for 512-D handwriting embeddings."""
    global _resnet_encoder
    if _resnet_encoder is not None:
        return _resnet_encoder
    try:
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    except Exception as exc:
        warnings.warn(f"Pretrained ImageNet weights unavailable ({exc}); falling back to uninitialized ResNet18.")
        model = models.resnet18(weights=None)
    model.fc = nn.Identity()
    model.eval().to(DEVICE)
    _resnet_encoder = model
    return _resnet_encoder


def extract_resnet_embedding(image: np.ndarray) -> np.ndarray:
    """Compute a 512-D L2-normalized embedding averaged over segmented line crops."""
    encoder = load_resnet18_encoder()
    crops = segment_lines(image) or [image]
    tensors = []
    for crop in crops[:12]:
        rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB) if crop.ndim == 3 else cv2.cvtColor(crop, cv2.COLOR_GRAY2RGB)
        tensors.append(_resnet_transform(Image.fromarray(rgb)))
    batch = torch.stack(tensors).to(DEVICE)
    with torch.no_grad():
        feats = encoder(batch).cpu().numpy()
    mean_vec = feats.mean(axis=0)
    norm = np.linalg.norm(mean_vec) + 1e-8
    return (mean_vec / norm).astype(np.float32)


ALPHABET_CHARS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789")

def build_character_prototype_bank(word_rows: pd.DataFrame) -> dict[str, dict]:
    """Build character prototypes from aligned IAM word crops without inventing unseen glyphs.

    Uses single-character word tokens directly and extracts proportional connected-component
    windows when multi-character words match the number of ink components.
    """
    per_char_vectors: dict[str, list[np.ndarray]] = {ch: [] for ch in ALPHABET_CHARS}

    for _, row in word_rows.iterrows():
        label = str(row["text"])
        if not label or not all(ch in per_char_vectors for ch in label):
            continue
        img = cv2.imread(row["image_path"])
        if img is None:
            continue

        if len(label) == 1:
            vf = extract_visual_handwriting_features(img)
            vec = np.array([
                vf["vis_cc_aspect_mean"],
                vf["vis_slant_angle_mean"],
                vf["vis_stroke_width_mean"],
                vf["vis_ink_density"],
                float(img.shape[0]),
            ], dtype=float)
            per_char_vectors[label].append(vec)
        else:
            # Uniform horizontal character alignment fallback for short clean words
            if len(label) <= 5 and img.shape[1] >= len(label) * 8:
                step = img.shape[1] / len(label)
                for idx, ch in enumerate(label):
                    x1 = int(round(idx * step))
                    x2 = int(round((idx + 1) * step))
                    patch = img[:, x1:x2]
                    if patch.size == 0:
                        continue
                    vf = extract_visual_handwriting_features(patch)
                    vec = np.array([
                        vf["vis_cc_aspect_mean"],
                        vf["vis_slant_angle_mean"],
                        vf["vis_stroke_width_mean"],
                        vf["vis_ink_density"],
                        float(patch.shape[0]),
                    ], dtype=float)
                    per_char_vectors[ch].append(vec)

    bank = {}
    for ch in ALPHABET_CHARS:
        samples = per_char_vectors[ch]
        if samples:
            bank[ch] = {
                "status": "available",
                "sample_count": len(samples),
                "prototype": np.mean(samples, axis=0),
            }
        else:
            bank[ch] = {"status": "missing", "sample_count": 0, "prototype": None}
    return bank


def compare_character_banks(bank_a: dict, bank_b: dict) -> dict:
    shared = [
        ch for ch in ALPHABET_CHARS
        if bank_a[ch]["status"] == "available" and bank_b[ch]["status"] == "available"
    ]
    if not shared:
        return {"shared_characters": 0, "coverage_ratio": 0.0, "char_visual_similarity": None}

    sims = []
    for ch in shared:
        va = bank_a[ch]["prototype"]
        vb = bank_b[ch]["prototype"]
        rel_diff = np.mean(np.abs(va - vb) / (np.abs(va) + np.abs(vb) + 1e-6))
        sims.append(float(np.clip(1.0 - rel_diff, 0.0, 1.0)))
    return {
        "shared_characters": len(shared),
        "coverage_ratio": round(len(shared) / len(ALPHABET_CHARS), 4),
        "char_visual_similarity": round(float(np.mean(sims)), 4),
    }

## 7. Writer-Disjoint Splits, Pairwise Dataset & 3-Way Ablation Study

We split documents strictly by `writer_id` **before** generating pairs so no writer or form leaks across train, validation, and test splits.
We then evaluate three configurations:
1. **NLP-Only Model**: 5 stylometric modules + TF-IDF character n-gram cosine similarity (fit on train split only).
2. **Visual-Only Model**: OpenCV stroke/slant/proportion differences + ResNet18 embedding cosine similarity.
3. **Multimodal Fusion**:
   - Weighted linear fusion: $S_{\text{final}} = \alpha S_{\text{NLP}} + \beta S_{\text{Visual}}$ (initial $\alpha=0.75, \beta=0.25$, plus validation-tuned weights)
   - Learned Logistic Regression meta-fusion

In [ ]:
def make_writer_disjoint_splits(
    df: pd.DataFrame,
    test_size: float = 0.25,
    val_size: float = 0.15,
    random_state: int = RANDOM_STATE,
):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    trainval_idx, test_idx = next(gss1.split(df, groups=df["writer_id"]))
    trainval = df.iloc[trainval_idx].reset_index(drop=True)
    test = df.iloc[test_idx].reset_index(drop=True)

    relative_val = val_size / (1.0 - test_size)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=relative_val, random_state=random_state + 1)
    train_idx, val_idx = next(gss2.split(trainval, groups=trainval["writer_id"]))
    train = trainval.iloc[train_idx].reset_index(drop=True)
    val = trainval.iloc[val_idx].reset_index(drop=True)
    return train, val, test


def attach_all_features(df: pd.DataFrame, use_resnet: bool = True) -> tuple[pd.DataFrame, np.ndarray | None]:
    """Extract NLP stylometric features and Visual features for every document in `df`."""
    rows = []
    resnet_vecs = []
    for _, row in df.iterrows():
        text_val = str(row.get("text", "") or "")
        if not text_val.strip():
            raise ValueError(f"Blank text for {row.get('doc_id')}. Provide GT text or run OCR first.")
        nlp_feats = extract_stylometric_features(text_val, line_count=int(row.get("line_count", 1)))
        img = load_page(row["image_path"])
        vis_feats = extract_visual_handwriting_features(img)
        rows.append({**nlp_feats, **vis_feats})
        if use_resnet:
            resnet_vecs.append(extract_resnet_embedding(img))

    feat_df = pd.DataFrame(rows).fillna(0.0)
    out_df = pd.concat([df.reset_index(drop=True), feat_df], axis=1)
    embs = np.vstack(resnet_vecs) if use_resnet and resnet_vecs else None
    return out_df, embs


def sample_pair_indices(df: pd.DataFrame, pairs_per_class: int | None = None, random_state: int = RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    by_writer = {w: g.index.to_numpy() for w, g in df.groupby("writer_id")}
    writers = list(by_writer.keys())

    pos = []
    for _, idxs in by_writer.items():
        if len(idxs) >= 2:
            for i in range(len(idxs)):
                for j in range(i + 1, len(idxs)):
                    pos.append((int(idxs[i]), int(idxs[j]), 1))

    neg = []
    for i, wa in enumerate(writers):
        for wb in writers[i + 1:]:
            for ia in by_writer[wa]:
                for ib in by_writer[wb]:
                    neg.append((int(ia), int(ib), 0))

    if not pos or not neg:
        raise ValueError("Need at least one writer with 2+ documents and at least 2 distinct writers.")

    n = min(len(pos), len(neg))
    if pairs_per_class is not None:
        n = min(n, pairs_per_class)

    pos_sel = [pos[i] for i in rng.choice(len(pos), size=n, replace=False)]
    neg_sel = [neg[i] for i in rng.choice(len(neg), size=n, replace=False)]
    selected = pos_sel + neg_sel
    rng.shuffle(selected)
    return selected


def build_pair_matrices(
    df: pd.DataFrame,
    pairs: list[tuple[int, int, int]],
    char_vectorizer: TfidfVectorizer,
    nlp_cols: list[str],
    vis_cols: list[str],
    resnet_embs: np.ndarray | None = None,
):
    tfidf_mat = char_vectorizer.transform(df["text"].fillna("").tolist())

    X_nlp, X_vis, y = [], [], []
    ia_list = [p[0] for p in pairs]
    ib_list = [p[1] for p in pairs]
    char_cos = 1.0 - paired_cosine_distances(tfidf_mat[ia_list], tfidf_mat[ib_list])

    nlp_mat = df[nlp_cols].to_numpy(dtype=float)
    vis_mat = df[vis_cols].to_numpy(dtype=float)

    for k, (ia, ib, label) in enumerate(pairs):
        nlp_diff = np.abs(nlp_mat[ia] - nlp_mat[ib])
        X_nlp.append(np.append(nlp_diff, char_cos[k]))

        vis_diff = np.abs(vis_mat[ia] - vis_mat[ib])
        if resnet_embs is not None:
            res_cos = float(np.dot(resnet_embs[ia], resnet_embs[ib]))
            X_vis.append(np.append(vis_diff, res_cos))
        else:
            X_vis.append(vis_diff)

        y.append(label)

    nlp_pair_cols = [*nlp_cols, "char_ngram_cosine"]
    vis_pair_cols = [*vis_cols, "resnet18_cosine"] if resnet_embs is not None else list(vis_cols)
    return np.asarray(X_nlp), np.asarray(X_vis), np.asarray(y), nlp_pair_cols, vis_pair_cols


# Run feature extraction on the IAM manifest
USE_RESNET = True
train_raw, val_raw, test_raw = make_writer_disjoint_splits(manifest)
train_df, train_embs = attach_all_features(train_raw, use_resnet=USE_RESNET)
val_df, val_embs = attach_all_features(val_raw, use_resnet=USE_RESNET)
test_df, test_embs = attach_all_features(test_raw, use_resnet=USE_RESNET)

# Fit character n-gram vectorizer ONLY on the training split
char_vectorizer = build_char_vectorizer()
char_vectorizer.fit(train_df["text"].fillna("").tolist())

train_pairs = sample_pair_indices(train_df, pairs_per_class=2000, random_state=RANDOM_STATE)
val_pairs = sample_pair_indices(val_df, pairs_per_class=1000, random_state=RANDOM_STATE + 1)
test_pairs = sample_pair_indices(test_df, pairs_per_class=1000, random_state=RANDOM_STATE + 2)

X_tr_nlp, X_tr_vis, y_train, nlp_pair_cols, vis_pair_cols = build_pair_matrices(
    train_df, train_pairs, char_vectorizer, ALL_NLP_FEATURE_COLS, VISUAL_FEATURE_COLS, train_embs
)
X_va_nlp, X_va_vis, y_val, _, _ = build_pair_matrices(
    val_df, val_pairs, char_vectorizer, ALL_NLP_FEATURE_COLS, VISUAL_FEATURE_COLS, val_embs
)
X_te_nlp, X_te_vis, y_test, _, _ = build_pair_matrices(
    test_df, test_pairs, char_vectorizer, ALL_NLP_FEATURE_COLS, VISUAL_FEATURE_COLS, test_embs
)

print("Train pairs:", len(y_train), "| Val pairs:", len(y_val), "| Test pairs:", len(y_test))
print("NLP pair dims:", X_tr_nlp.shape[1], "| Visual pair dims:", X_tr_vis.shape[1])

In [ ]:
def train_rf_pair_model(X_train, y_train):
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    )
    model.fit(X_train, y_train)
    return model


def compute_far_frr_eer(y_true: np.ndarray, probs: np.ndarray, threshold: float = 0.5) -> dict:
    pred = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    far = fp / max(1, fp + tn)
    frr = fn / max(1, fn + tp)

    fpr, tpr, thresholds = roc_curve(y_true, probs)
    fnr = 1.0 - tpr
    eer_idx = int(np.argmin(np.abs(fpr - fnr)))
    eer = float((fpr[eer_idx] + fnr[eer_idx]) / 2.0)
    eer_thresh = float(thresholds[eer_idx])
    return {"far": far, "frr": frr, "eer": eer, "eer_threshold": eer_thresh}


def evaluate_probs(name: str, y_true: np.ndarray, probs: np.ndarray, threshold: float = 0.5) -> dict:
    pred = (probs >= threshold).astype(int)
    bio = compute_far_frr_eer(y_true, probs, threshold=threshold)
    return {
        "model": name,
        "decision_threshold": round(threshold, 4),
        "accuracy": round(accuracy_score(y_true, pred), 4),
        "precision": round(precision_score(y_true, pred, zero_division=0), 4),
        "recall": round(recall_score(y_true, pred, zero_division=0), 4),
        "f1": round(f1_score(y_true, pred, zero_division=0), 4),
        "roc_auc": round(roc_auc_score(y_true, probs), 4),
        "far": round(bio["far"], 4),
        "frr": round(bio["frr"], 4),
        "eer": round(bio["eer"], 4),
    }


nlp_model = train_rf_pair_model(X_tr_nlp, y_train)
vis_model = train_rf_pair_model(X_tr_vis, y_train)

# Validation probabilities for tuning fusion & threshold
p_val_nlp = nlp_model.predict_proba(X_va_nlp)[:, 1]
p_val_vis = vis_model.predict_proba(X_va_vis)[:, 1]

fusion_meta = LogisticRegression(random_state=RANDOM_STATE)
fusion_meta.fit(np.column_stack([p_val_nlp, p_val_vis]), y_val)

# Tune weighted alpha on validation ROC-AUC
best_alpha, best_val_auc = DEFAULT_ALPHA, -1.0
for alpha in np.linspace(0.0, 1.0, 21):
    p_mix = alpha * p_val_nlp + (1.0 - alpha) * p_val_vis
    auc = roc_auc_score(y_val, p_mix)
    if auc > best_val_auc:
        best_val_auc = auc
        best_alpha = float(alpha)
best_beta = round(1.0 - best_alpha, 4)

# Calibrate decision threshold on validation EER using weighted fusion
p_val_tuned = best_alpha * p_val_nlp + best_beta * p_val_vis
calibrated_threshold = compute_far_frr_eer(y_val, p_val_tuned)["eer_threshold"]

# Evaluate on held-out unseen writers/forms (test split)
p_test_nlp = nlp_model.predict_proba(X_te_nlp)[:, 1]
p_test_vis = vis_model.predict_proba(X_te_vis)[:, 1]
p_test_default = DEFAULT_ALPHA * p_test_nlp + DEFAULT_BETA * p_test_vis
p_test_tuned = best_alpha * p_test_nlp + best_beta * p_test_vis
p_test_meta = fusion_meta.predict_proba(np.column_stack([p_test_nlp, p_test_vis]))[:, 1]

ablation_df = pd.DataFrame([
    evaluate_probs("1. NLP-only", y_test, p_test_nlp, threshold=0.5),
    evaluate_probs("2. Visual-only", y_test, p_test_vis, threshold=0.5),
    evaluate_probs(f"3a. Weighted Fusion ({DEFAULT_ALPHA:.2f}/{DEFAULT_BETA:.2f})", y_test, p_test_default, threshold=0.5),
    evaluate_probs(f"3b. Tuned Fusion ({best_alpha:.2f}/{best_beta:.2f})", y_test, p_test_tuned, threshold=calibrated_threshold),
    evaluate_probs("3c. Learned Meta-Fusion", y_test, p_test_meta, threshold=0.5),
])

display(ablation_df)

fig, ax = plt.subplots(figsize=(7, 5))
for label, probs in [
    ("NLP-only", p_test_nlp),
    ("Visual-only", p_test_vis),
    (f"Weighted Fusion ({DEFAULT_ALPHA:.2f}/{DEFAULT_BETA:.2f})", p_test_default),
    ("Learned Meta-Fusion", p_test_meta),
]:
    RocCurveDisplay.from_predictions(y_test, probs, name=label, ax=ax)
ax.set_title("Check Mate Ablation Study — Held-Out Test ROC")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Topic-Controlled Hard-Negative Evaluation
In IAM, several passages (`a01-000`, `a01-003`, ...) were transcribed by multiple different writers (`u`, `x`, etc.).
Comparing `a01-000u` vs `a01-000x` holds the topic/text nearly constant so we can verify whether the visual branch catches different writers even when linguistic content overlaps.

In [ ]:
def evaluate_same_passage_hard_negatives(df: pd.DataFrame, resnet_embs: np.ndarray | None = None):
    groups = df.groupby("passage_id")
    hard_pairs = []
    for passage, g in groups:
        forms = g["form_id"].unique()
        if len(forms) >= 2:
            sub_a = g[g["form_id"] == forms[0]].iloc[0]
            sub_b = g[g["form_id"] == forms[1]].iloc[0]
            hard_pairs.append((int(sub_a.name), int(sub_b.name), 0))
    if not hard_pairs:
        print("No multi-form passages found in this split.")
        return None
    X_hn_nlp, X_hn_vis, y_hn, _, _ = build_pair_matrices(
        df, hard_pairs, char_vectorizer, ALL_NLP_FEATURE_COLS, VISUAL_FEATURE_COLS, resnet_embs=resnet_embs
    )
    p_nlp = nlp_model.predict_proba(X_hn_nlp)[:, 1]
    p_vis = vis_model.predict_proba(X_hn_vis)[:, 1]
    p_fus = DEFAULT_ALPHA * p_nlp + DEFAULT_BETA * p_vis
    print(f"Hard same-passage negative pairs evaluated: {len(hard_pairs)}")
    print(f"Mean predicted compatibility -> NLP-only: {p_nlp.mean():.3f} | Visual-only: {p_vis.mean():.3f} | Fusion: {p_fus.mean():.3f}")
    return pd.DataFrame({"p_nlp": p_nlp, "p_visual": p_vis, "p_fusion": p_fus})

_ = evaluate_same_passage_hard_negatives(test_df, resnet_embs=test_embs)

## 8. Explainability, Insufficient-Evidence Guard & Two-Image Verification Demo

Let's inspect the most informative NLP & visual pair-difference features using permutation importance, and run a complete end-to-end verification function matching the Check Mate API response contract.

In [ ]:
perm_nlp = permutation_importance(
    nlp_model, X_te_nlp, y_test, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1
)
top_nlp = (
    pd.DataFrame({"feature": nlp_pair_cols, "importance": perm_nlp.importances_mean})
    .sort_values("importance", ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top_nlp["feature"][::-1], top_nlp["importance"][::-1], color="#1689F5")
ax.set_title("Top 15 NLP Pairwise Features (Permutation Importance)")
ax.set_xlabel("Mean Accuracy Decrease")
plt.tight_layout()
plt.show()


def verify_handwriting_pair(
    reference_image_path: str | Path,
    comparison_image_path: str | Path,
    reference_text_override: str | None = None,
    comparison_text_override: str | None = None,
    alpha: float = DEFAULT_ALPHA,
    beta: float = DEFAULT_BETA,
    threshold: float = calibrated_threshold,
) -> dict:
    """Run full Check Mate verification (OCR or text override -> 5 NLP modules -> Visual -> Fusion)."""
    ref_img = load_page(reference_image_path)
    cmp_img = load_page(comparison_image_path)

    if reference_text_override is None:
        ref_ocr = transcribe_page(reference_image_path)
        ref_text, ref_lines = ref_ocr["text"], ref_ocr["line_count"]
    else:
        ref_text = reference_text_override
        ref_lines = max(1, len([ln for ln in ref_text.splitlines() if ln.strip()]))

    if comparison_text_override is None:
        cmp_ocr = transcribe_page(comparison_image_path)
        cmp_text, cmp_lines = cmp_ocr["text"], cmp_ocr["line_count"]
    else:
        cmp_text = comparison_text_override
        cmp_lines = max(1, len([ln for ln in cmp_text.splitlines() if ln.strip()]))

    ref_nlp = extract_stylometric_features(ref_text, line_count=ref_lines)
    cmp_nlp = extract_stylometric_features(cmp_text, line_count=cmp_lines)

    warnings_list = [
        "OCR errors can affect linguistic features.",
        "This analysis estimates stylistic compatibility and is not forensic proof of identity.",
    ]

    if ref_nlp["meta_word_count"] < MIN_WORDS_FOR_VERIFICATION or cmp_nlp["meta_word_count"] < MIN_WORDS_FOR_VERIFICATION:
        return {
            "status": "insufficient_evidence",
            "message": f"At least {MIN_WORDS_FOR_VERIFICATION} words per sample are required for reliable stylometry.",
            "nlp_score": None,
            "visual_score": None,
            "combined_score": None,
            "reference_text": ref_text,
            "comparison_text": cmp_text,
            "warnings": warnings_list,
        }

    ref_vec = np.array([ref_nlp[c] for c in ALL_NLP_FEATURE_COLS], dtype=float)
    cmp_vec = np.array([cmp_nlp[c] for c in ALL_NLP_FEATURE_COLS], dtype=float)
    tfidf_pair = char_vectorizer.transform([ref_text, cmp_text])
    char_cos = float(1.0 - paired_cosine_distances(tfidf_pair[0], tfidf_pair[1])[0])
    x_nlp = np.append(np.abs(ref_vec - cmp_vec), char_cos).reshape(1, -1)

    ref_vis = extract_visual_handwriting_features(ref_img)
    cmp_vis = extract_visual_handwriting_features(cmp_img)
    ref_vvec = np.array([ref_vis[c] for c in VISUAL_FEATURE_COLS], dtype=float)
    cmp_vvec = np.array([cmp_vis[c] for c in VISUAL_FEATURE_COLS], dtype=float)
    if USE_RESNET:
        res_cos = float(np.dot(extract_resnet_embedding(ref_img), extract_resnet_embedding(cmp_img)))
        x_vis = np.append(np.abs(ref_vvec - cmp_vvec), res_cos).reshape(1, -1)
    else:
        x_vis = np.abs(ref_vvec - cmp_vvec).reshape(1, -1)

    nlp_score = float(nlp_model.predict_proba(x_nlp)[0, 1])
    vis_score = float(vis_model.predict_proba(x_vis)[0, 1])
    combined = float(alpha * nlp_score + beta * vis_score)
    status = "compatible" if combined >= threshold else "not_compatible"

    module_breakdown = compute_nlp_module_similarities(ref_nlp, cmp_nlp, char_ngram_cosine=char_cos)
    feature_comparison = [
        {
            "feature": c,
            "reference": round(float(ref_nlp[c]), 4),
            "comparison": round(float(cmp_nlp[c]), 4),
            "absolute_difference": round(abs(float(ref_nlp[c]) - float(cmp_nlp[c])), 4),
        }
        for c in ["avg_word_length", "avg_sentence_length", "ttr", "mattr_50", "punct_per_token", "noun_to_verb_ratio", "dependency_depth_mean"]
    ]

    return {
        "status": status,
        "message": f"Verification complete ({status}, threshold={threshold:.3f}).",
        "nlp_score": round(nlp_score, 4),
        "visual_score": round(vis_score, 4),
        "combined_score": round(combined, 4),
        "nlp_module_scores": module_breakdown,
        "reference_text": ref_text,
        "comparison_text": cmp_text,
        "feature_comparison": feature_comparison,
        "warnings": warnings_list,
    }


# Smoke test on a held-out test form pair (ground-truth text mode; omit overrides to run TrOCR)
demo_form = test_df["form_id"].iloc[0]
demo_rows = test_df[test_df["form_id"] == demo_form]
if len(demo_rows) >= 2:
    demo_res = verify_handwriting_pair(
        demo_rows.iloc[0]["image_path"],
        demo_rows.iloc[1]["image_path"],
        reference_text_override=demo_rows.iloc[0]["text"],
        comparison_text_override=demo_rows.iloc[1]["text"],
    )
    print(json.dumps({k: v for k, v in demo_res.items() if k not in {"reference_text", "comparison_text"}}, indent=2))

# Character-level prototype bank demo on the same form's words
form_words = df_words[df_words["form_id"] == demo_form]
mid_line = sorted(form_words["line_no"].unique())[len(form_words["line_no"].unique()) // 2]
bank_a = build_character_prototype_bank(form_words[form_words["line_no"] < mid_line])
bank_b = build_character_prototype_bank(form_words[form_words["line_no"] >= mid_line])
print("Character Prototype Bank comparison:", compare_character_banks(bank_a, bank_b))

## 9. Save Artifacts for Backend Integration

Saves the exact trained classifiers, feature schemas, TF-IDF character vectorizer, and calibrated fusion configuration to `../backend/artifacts/` so FastAPI loads identical preprocessing and feature ordering.

In [ ]:
def save_check_mate_artifacts(artifact_dir: Path = ARTIFACT_DIR):
    artifact_dir.mkdir(parents=True, exist_ok=True)
    dump(nlp_model, artifact_dir / "nlp_pair_classifier.joblib")
    dump(nlp_pair_cols, artifact_dir / "nlp_feature_columns.joblib")
    dump(char_vectorizer, artifact_dir / "char_tfidf_vectorizer.joblib")
    dump(vis_model, artifact_dir / "visual_pair_classifier.joblib")
    dump(vis_pair_cols, artifact_dir / "visual_feature_columns.joblib")
    dump(fusion_meta, artifact_dir / "fusion_meta_model.joblib")

    config = {
        "alpha": DEFAULT_ALPHA,
        "beta": DEFAULT_BETA,
        "tuned_alpha": best_alpha,
        "tuned_beta": best_beta,
        "calibrated_threshold": float(calibrated_threshold),
        "min_words_for_verification": MIN_WORDS_FOR_VERIFICATION,
        "use_resnet": USE_RESNET,
        "writer_id_source": str(manifest["writer_id_source"].iloc[0]),
        "ablation_metrics": ablation_df.to_dict(orient="records"),
    }
    (artifact_dir / "fusion_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    print("Saved Check Mate artifacts to:", artifact_dir.resolve())

save_check_mate_artifacts()